# Full pipeline v3 on the G4 machine (48 cores, 176 GB RAM, RTX PRO 6000)

Every train S1 entity goes through retrieval, the candidate filter, the cross-encoder and feature computation,
exactly like test, so the competition features match at train and test time.

**Drive layout needed** (no code upload; the code is embedded in cell 2):
```
My Drive/ber_ce/ce_model/                trained cross-encoder
My Drive/ber_v3/parquet/*.parquet        7 raw data files (work/parquet on the laptop)
My Drive/ber_v3/work_ids_ce10.parquet    (work/cand on the laptop)
```
Run the cells top to bottom and **save the notebook after each cell** so progress is visible on the laptop.
Rough timings: setup 5 min, prepare + candidates ~60 min, filter ~25 min, CE scoring ~40 min (background, while
features compute), stage 1 ~25 min, stage 2 ~30 min, predict ~10 min.


In [1]:
!nproc; free -g | head -2; nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/ber_v3'
CE_DRIVE = '/content/drive/MyDrive/ber_ce/ce_model'
!ls -la {DRIVE} {DRIVE}/parquet {CE_DRIVE}

Mounted at /content/drive
/content/drive/MyDrive/ber_ce/ce_model:
total 1102843
drwx------ 2 root root       4096 Sep 26 12:41 .
drwx------ 3 root root       4096 Sep 26 11:42 ..
-rw------- 1 root root        840 Sep 26 12:41 config.json
-rw------- 1 root root        198 Sep 26 12:41 metrics.json
-rw------- 1 root root 1112201932 Sep 26 12:41 model.safetensors
-rw------- 1 root root        343 Sep 26 12:41 tokenizer_config.json
-rw------- 1 root root   17098086 Sep 26 12:41 tokenizer.json

/content/drive/MyDrive/ber_v3:
total 1638
drwx------ 3 root root    4096 Sep 26 14:12 .
drwx------ 6 root root    4096 Sep 26 18:57 ..
drwx------ 2 root root    4096 Sep 26 14:12 parquet
-rw------- 1 root root 1665015 Sep 26 11:40 work_ids_ce10.parquet

/content/drive/MyDrive/ber_v3/parquet:
total 1642618
drwx------ 2 root root      4096 Sep 26 14:12 .
drwx------ 3 root root      4096 Sep 26 14:12 ..
-rw------- 1 root root 110464226 Sep 26 08:39 test_source1.parquet
-rw------- 1 root root 336017967 S

In [2]:
CODE = '...'

In [3]:
# Unpack the pipeline code embedded in the cell above (rebuilt from the repo by colab/build_notebook.py).
import base64, io, zipfile, shutil, os
shutil.rmtree('/content/code', ignore_errors=True)
zipfile.ZipFile(io.BytesIO(base64.b64decode(CODE))).extractall('/content/code')
print(sorted(os.listdir('/content/code/src')))

['__init__.py', 'bench_translit.py', 'candidates.py', 'ce_data.py', 'config.py', 'context.py', 'cross_encoder.py', 'data.py', 'decode.py', 'eda.py', 'eval_retrieval.py', 'features.py', 'filter_sweep.py', 'filtering.py', 'lgbutil.py', 'metric.py', 'normalize.py', 'pipeline_v3.py', 'predict.py', 'prepare.py', 'retrieve.py', 'submission.py', 'text.py', 'train.py', 'train2.py', 'translit.py']


In [4]:
# Setup: data, cross-encoder model, and a separate environment with the pinned packages.
import os, sys
!mkdir -p /content/work/parquet /content/work/cand /content/work/output
!cp -n {DRIVE}/parquet/*.parquet /content/work/parquet/ && cp -n {DRIVE}/work_ids_ce10.parquet /content/work/cand/ && ls -la /content/work/parquet
!rm -rf /content/work/ce_model && cp -r {CE_DRIVE} /content/work/ce_model && ls /content/work/ce_model
!pip install -q uv && uv venv -q --allow-existing /content/venv && uv pip install -q --python /content/venv/bin/python -r /content/code/requirements.txt
os.environ.update({'BER_WORK_DIR': '/content/work', 'BER_OUTPUT_DIR': '/content/work/output', 'BER_THREADS': '46',
                   'BER_CE_MODEL': '/content/work/ce_model', 'BER_CE_PYTHON': sys.executable,
                   'PYTHONIOENCODING': 'utf-8', 'BER_K': '20', 'BER_CAND_BUDGET': '16', 'BER_MODEL_FRAC': '0.3'})
PY = '/content/venv/bin/python'
!{PY} -c "import pandas, lightgbm, sparse_dot_topn, rapidfuzz; print('pipeline env ok', pandas.__version__, lightgbm.__version__)"

cp: warning: behavior of -n is non-portable and may change in future; use --update=none instead
cp: warning: behavior of -n is non-portable and may change in future; use --update=none instead
total 1642644
drwxr-xr-x 2 root root      4096 Sep 26 18:58 .
drwxr-xr-x 5 root root      4096 Sep 26 18:57 ..
-rw------- 1 root root 110464226 Sep 26 18:57 test_source1.parquet
-rw------- 1 root root 336017967 Sep 26 18:58 test_source2.parquet
-rw------- 1 root root 335537737 Sep 26 18:58 test_source3.parquet
-rw------- 1 root root  92419561 Sep 26 18:58 train_ground_truth.parquet
-rw------- 1 root root 136677134 Sep 26 18:58 train_source1.parquet
-rw------- 1 root root 332018842 Sep 26 18:58 train_source2.parquet
-rw------- 1 root root 338894978 Sep 26 18:58 train_source3.parquet
config.json   model.safetensors      tokenizer.json
metrics.json  tokenizer_config.json
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.5/20.5 MB 179.7 MB/s eta 0:00:0000:0100:01
pipeline env ok 3.0.6 4.7.0


In [5]:
# 0. GPU, in the background: train the larger cross-encoder (bge-reranker-v2-m3, Apache-2.0, 568M) while the CPUs
#    run search. It later rescores only the pairs the base model is unsure about. Optional: skipped if it fails.
!mkdir -p /content/ce_large && cp -n /content/drive/MyDrive/ber_ce/ce_pairs.parquet /content/drive/MyDrive/ber_ce/records_train.parquet /content/ce_large/
!cd /content/code && nohup {sys.executable} src/cross_encoder.py train --data /content/ce_large --model BAAI/bge-reranker-v2-m3 --out /content/work/ce_large_model --batch 128 --lr 2e-5 --max-train-pairs 1200000 --workers 8 > /content/work/ce_large_train.log 2>&1 &
!sleep 240; tail -4 /content/work/ce_large_train.log; nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv

cp: warning: behavior of -n is non-portable and may change in future; use --update=none instead
  "heldout_logloss": 0.0226657991863271,
  "minutes": 28.363339006900787
}
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.17s/it]
utilization.gpu [%], memory.used [MiB]
0 %, 0 MiB


In [6]:
# 1. Normalize every record, then retrieval candidates for ALL train and test S1 entities (top-20 per route).
!cd /content/code && {PY} -m src.pipeline_v3 prepare candidates 2>&1 | tee -a /content/work/v3.log

19:32:11 === prepare ===
train source1: 2,206,821 rows in 7s
train source2: 5,034,616 rows in 15s
train source3: 5,285,603 rows in 16s
test source1: 1,732,544 rows in 6s
test source2: 4,887,273 rows in 15s
test source3: 5,082,316 rows in 16s
19:33:29 === candidates ===
train/all India S2: 883,188 queries x 2,017,799 docs -> 47,711,938 pairs (54.0/query) in 242s
train/all US S2: 1,323,633 queries x 3,016,817 docs -> 75,480,766 pairs (57.0/query) in 294s
train/all India S3: 883,188 queries x 2,115,547 docs -> 48,998,080 pairs (55.5/query) in 238s
train/all US S3: 1,323,633 queries x 3,170,056 docs -> 75,496,520 pairs (57.0/query) in 312s
test/all France S2: 259,452 queries x 703,378 docs -> 15,236,078 pairs (58.7/query) in 34s
test/all India S2: 809,986 queries x 2,312,565 docs -> 43,326,592 pairs (53.5/query) in 263s
test/all US S2: 663,106 queries x 1,871,330 docs -> 36,951,769 pairs (55.7/query) in 119s
test/all France S3: 259,452 queries x 731,615 docs -> 15,262,094 pairs (58.8/query

In [7]:
# 2. Candidate filter: trained on model entities, features over the whole population; prints the size/ceiling sweep.
!cd /content/code && {PY} -m src.pipeline_v3 filter 2>&1 | tee -a /content/work/v3.log

20:05:24 === filter ===
20:06:21 filter features train/India_s2: 47,711,938 pairs | 56s
20:07:11 filter features train/India_s3: 48,998,080 pairs | 107s
20:08:33 filter features train/US_s2: 75,480,766 pairs | 189s
20:09:55 filter features train/US_s3: 75,496,520 pairs | 271s
20:11:17 filter model: 1272 rounds | 352s
20:23:36 validation: all retrieved 112.2/entity | chosen {'tau': 0.001, 'r': 99.0, 'k_max': 15.0} -> 15.34/entity, p95 29, oracle ceiling 0.9948
   tau  r  k_max  mean_cands  p95   oracle
0.0005 99     15   18.564702 30.0 0.995049
0.0010 99     15   15.339273 29.0 0.994835
0.0005 99     12   17.071638 24.0 0.994771
0.0010 99     12   14.497833 24.0 0.994613
0.0005 99     10   15.616178 20.0 0.994465
0.0020 99     15   12.481996 25.0 0.994443
0.0010 99     10   13.583651 20.0 0.994357
0.0020 99     12   12.097179 23.0 0.994307
0.0020 99     10   11.585258 20.0 0.994109
0.0005 99      8   13.653942 16.0 0.994023
0.0010 99      8   12.252384 16.0 0.993953
0.0020 99      8   1

In [8]:
# 3. Export texts for the cross-encoder, then score every kept pair on the GPU IN THE BACKGROUND.
!cd /content/code && {PY} -m src.pipeline_v3 ce_export 2>&1 | tee -a /content/work/v3.log
import subprocess
subprocess.Popen(f'cd /content/code && exec {PY} -m src.pipeline_v3 ce_score > /content/work/ce_score.log 2>&1', shell=True, start_new_session=True)
!sleep 90; tail -3 /content/work/ce_score.log; nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv


20:35:42 === ce_export ===
20:37:02 ce export train: 33,868,758 pairs | 80s
20:38:16 ce export test: 31,716,731 pairs | 154s
20:38:16 === ce_score ===
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 15833.80it/s]
utilization.gpu [%], memory.used [MiB]
99 %, 35311 MiB


In [9]:
# 4. Pair features on the CPU while the GPU scores.
!cd /content/code && {PY} -m src.pipeline_v3 features 2>&1 | tee -a /content/work/v3.log

20:39:47 === features ===
20:45:48 features train: (33868758, 52) | 362s
20:51:26 features test: (31716731, 51) | 699s


In [10]:
# 5. Wait for the cross-encoder to finish (prints its log).
!until grep -qE "CE DONE|Traceback|Error" /content/work/ce_score.log; do sleep 30; done; tail -8 /content/work/ce_score.log

20:38:16 === ce_score ===
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 15833.80it/s]
scored 33,868,758 pairs in 44.4 min -> /content/work/v3/ce/train_scores.parquet
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 14351.10it/s]
scored 31,716,731 pairs in 42.4 min -> /content/work/v3/ce/test_scores.parquet
22:05:41 CE DONE


In [11]:
# 5b. Wait for the large cross-encoder to finish training (its files are saved last), then rescore uncertain pairs.
!until [ -f /content/work/ce_large_model/metrics.json ] || grep -qE "Traceback|Error" /content/work/ce_large_train.log; do sleep 30; done; tail -12 /content/work/ce_large_train.log
!cd /content/code && {PY} -m src.pipeline_v3 ce_large 2>&1 | tee -a /content/work/v3.log

epoch 0 step 8800/9375 loss 0.0332 | 711 pairs/s
epoch 0 step 9000/9375 loss 0.0120 | 711 pairs/s
epoch 0 step 9200/9375 loss 0.0499 | 711 pairs/s
{
  "model": "BAAI/bge-reranker-v2-m3",
  "train_pairs": 1200000,
  "heldout_pairs": 52156,
  "heldout_auc": 0.99968014908775,
  "heldout_logloss": 0.0226657991863271,
  "minutes": 28.363339006900787
}
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.17s/it]
22:06:38 === ce_large ===
22:06:38 large cross-encoder train: 1,964,352 uncertain pairs (5.8%)
Loading weights: 100%|██████████| 393/393 [00:00<00:00, 14493.26it/s]
scored 1,964,352 pairs in 6.1 min -> /content/work/v3/ce/train_large_scores.parquet
22:12:58 large cross-encoder test: 1,630,693 uncertain pairs (5.1%)
Loading weights: 100%|██████████| 393/393 [00:00<00:00, 15408.56it/s]
scored 1,630,693 pairs in 5.3 min -> /content/work/v3/ce/test_large_scores.parquet
22:18:31 CE LARGE DONE


In [12]:
# 6. Stage 1 (5-fold out-of-fold on model entities), stage 2 (context features, validation report), test predictions.
!cd /content/code && {PY} -m src.pipeline_v3 stage1 stage2 predict 2>&1 | tee -a /content/work/v3.log

22:18:32 === stage1 ===
22:18:35 stage 1: 33,868,758 train pairs, 9,135,726 from model entities, 50 features, cross-encoder coverage 100.0%
22:19:20 stage 1 fold 0: 656 rounds | 49s
22:20:05 stage 1 fold 1: 659 rounds | 94s
22:20:50 stage 1 fold 2: 650 rounds | 138s
22:21:39 stage 1 fold 3: 718 rounds | 187s
22:22:20 stage 1 fold 4: 590 rounds | 228s
22:24:07 stage 1 done | 336s
22:24:07 === stage2 ===
22:30:18 train context features (33868758, 24) | 370s
22:30:44 stage 2 validation model: 251 rounds | 397s
  stage1 + one-owner     threshold 0.65 -> 0.9845 | expected-F -> 0.9840 (non-singleton 0.9846, singleton acc 0.973)
  stage2 + one-owner     threshold 0.75 -> 0.9848 | expected-F -> 0.9853 (non-singleton 0.9848, singleton acc 0.993)
    stage 2 India        threshold 0.65 -> 0.9825 | expected-F -> 0.9829 (non-singleton 0.9823, singleton acc 0.993)
    stage 2 US           threshold 0.75 -> 0.9864 | expected-F -> 0.9869 (non-singleton 0.9865, singleton acc 0.993)
22:38:27 stage 2 do

In [13]:
# 7. Copy the submission files and reports to Drive.
!mkdir -p {DRIVE}/out && cp /content/work/output/*.tsv /content/work/v3/validation.json /content/work/v3/filter_sweep.csv /content/work/v3/filter_op.json /content/work/v3.log {DRIVE}/out/ && ls -la {DRIVE}/out

total 515211
drwx------ 2 root root      4096 Sep 26 22:42 .
drwx------ 4 root root      4096 Sep 26 22:42 ..
-rw------- 1 root root 431136015 Sep 26 22:42 candidate_pairs.tsv
-rw------- 1 root root       104 Sep 26 22:42 filter_op.json
-rw------- 1 root root      6325 Sep 26 22:42 filter_sweep.csv
-rw------- 1 root root  96417826 Sep 26 22:42 matching_results.tsv
-rw------- 1 root root      4795 Sep 26 22:42 v3.log
-rw------- 1 root root       686 Sep 26 22:42 validation.json


In [1]:
CODE4 = '...'
import base64, io, zipfile
zipfile.ZipFile(io.BytesIO(base64.b64decode(CODE4))).extractall('/content/code')
# OVERNIGHT, unattended. v4: both cross-encoders retrained on v3's own candidate pairs (two folds, so every score
# stays out-of-sample), then stages 1-2 again. v5: a 4B LLM reranker (Qwen3-Reranker-4B + LoRA) on the pairs both
# cross-encoders are unsure about, then stages 1-2 again. Logs: /content/work/v4.log and v5.log.
import os, sys, time, subprocess
os.environ['BER_CE_PYTHON'] = sys.executable
!pip install -q peft
!mkdir -p {DRIVE}/out_v3_models && cp /content/work/v3/*.txt /content/work/v3/*.json {DRIVE}/out_v3_models/ 2>/dev/null

def run(steps, logfile):
    cmd = f'cd /content/code && {PY} -m src.pipeline_v4 {steps} >> {logfile} 2>&1'
    ok = subprocess.run(cmd, shell=True).returncode == 0
    print(time.strftime('%H:%M:%S'), 'OK' if ok else 'FAILED', steps, flush=True)
    return ok

def save(name, logfile):
    subprocess.run(f'mkdir -p {DRIVE}/{name} && cp /content/work/output/*.tsv /content/work/v3/validation.json '
                   f'/content/work/v3/val_pairs.parquet {logfile} {DRIVE}/{name}/', shell=True)
    print(time.strftime('%H:%M:%S'), 'saved to Drive:', name, sorted(os.listdir(f'{DRIVE}/{name}')), flush=True)

if run('ce_data ce_train ce_score ce_large stage1 stage2 predict', '/content/work/v4.log'):
    save('out_v4', '/content/work/v4.log')
    if run('llm_data llm_train llm_score stage1 stage2 predict', '/content/work/v5.log'):
        save('out_v5', '/content/work/v5.log')

# Save the expensive state to Drive (kept pairs, every model score, small model files; the feature files rebuild in
# ~15 min), then release the runtime so it stops using compute units. A later run restores it from Drive.
import glob, shutil
for src in glob.glob('/content/work/v3/**/*', recursive=True) + glob.glob('/content/work/v4/llm*/**/*', recursive=True):
    if os.path.isfile(src) and not os.path.basename(src).startswith('features_'):
        dst = os.path.join(DRIVE, 'state', os.path.relpath(src, '/content/work'))
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy(src, dst)
for f in ('/content/work/v4.log', '/content/work/v5.log'):
    if os.path.exists(f):
        shutil.copy(f, DRIVE)
print(time.strftime('%H:%M:%S'), 'state saved to Drive; releasing the runtime', flush=True)
from google.colab import runtime
runtime.unassign()


NameError: name 'PY' is not defined